In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np
import time
import random
import os

In [ ]:
# now we will collect the data we need from urls we collected earlier
# data like car name, year, mileage, etc. also and our target price

In [ ]:
df = pd.read_csv("car_links_unique.csv")
checkpoint_file = "car_ads_checkpoint.csv"

# Resume from checkpoint if exists
if os.path.exists(checkpoint_file):
    result_df = pd.read_csv(checkpoint_file)
    done_urls = set(result_df['url'])
    results = result_df.to_dict('records')
else:
    results = []
    done_urls = set()

for i, url in enumerate(df['car_link']):
    if url in done_urls:
        continue
    try:
        response = requests.get(url, timeout=30)
        soup = BeautifulSoup(response.text, "html.parser")

        def get_text_by_label(label):
            el = soup.find("span", string=label)
            if el:
                val = el.find_next("span")
                return val.get_text(strip=True) if val else np.nan
            return np.nan

        brand = get_text_by_label("Brand")
        model = get_text_by_label("Model")
        kilometers = get_text_by_label("Kilometers")
        year = get_text_by_label("Year")
        fuel_type = get_text_by_label("Fuel Type")
        transmission = get_text_by_label("Transmission Type")
        engine_capacity = get_text_by_label("Engine Capacity (CC)")
        body_type = get_text_by_label("Body Type")

        desc_div = soup.find("div", {"aria-label": "Description"})
        description = desc_div.get_text(strip=True) if desc_div else np.nan

        price_span = soup.find("span", {"aria-label": "Price"})
        price = price_span.get_text(strip=True) if price_span else np.nan

        results.append({
            "url": url,
            "Brand": brand,
            "Model": model,
            "Kilometers": kilometers,
            "Year": year,
            "Fuel Type": fuel_type,
            "Transmission Type": transmission,
            "Engine Capacity (CC)": engine_capacity,
            "Body Type": body_type,
            "Description": description,
            "Price_EGP": price
        })
    except Exception as e:
        print(f"Error for URL {url}: {e}")
        results.append({
            "url": url,
            "Brand": np.nan,
            "Model": np.nan,
            "Kilometers": np.nan,
            "Year": np.nan,
            "Fuel Type": np.nan,
            "Transmission Type": np.nan,
            "Engine Capacity (CC)": np.nan,
            "Body Type": np.nan,
            "Description": np.nan,
            "Price_EGP": np.nan
        })
    print(f"Processed {i+1}/{len(df['car_link'])} URLs")
    time.sleep(random.randint(2, 5))

    # Save every 2 URLs
    if (i + 1) % 2 == 0:
        pd.DataFrame(results).to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved at {i+1} URLs")

# Final save
pd.DataFrame(results).to_csv("car_ads_details.csv", index=False)
print("Scraping complete. Data saved to car_ads_details.csv")

In [3]:
df=pd.read_csv("car_ads_details.csv")

In [4]:
df.head(10)

,url,Brand,Model,Kilometers,Year,Fuel Type,Transmission Type,Engine Capacity (CC),Body Type,Description,Price_EGP
0,https://www.dubizzle.com.eg/en/ad/%D9%81%D9%88...,Ford,Escort,130000,1998.0,Benzine,Manual,1300.0,Sedan,Descriptionفورد سكورت 98 مرور الاسماعيليه\n13...,"EGP 140,000"
1,https://www.dubizzle.com.eg/en/ad/%D8%B3%D8%A8...,Speranza,A113,83000,2012.0,Benzine,Manual,1300.0,Hatchback,Descriptionسياره زيرو بالكامل من المالك مباشره...,"EGP 243,000"
2,https://www.dubizzle.com.eg/en/ad/%D9%87%D9%88...,Honda,City,100000,2008.0,Benzine,Automatic,1500.0,Sedan,Descriptionسيارة هوندا سيتي موديل 2008 راشه با...,"EGP 240,000"
3,https://www.dubizzle.com.eg/en/ad/%D9%81%D9%88...,Volkswagen,Pointer,180000,2006.0,Benzine,Manual,1600.0,Hatchback,Descriptionعربيه فولكس بونتير موديل ٢٠٠٦ مرور ...,"EGP 155,000"
4,https://www.dubizzle.com.eg/en/ad/geely-emgran...,Geely,Emgrand 7,215000,2017.0,Diesel,Manual,1600.0,Sedan,Descriptionسيارة فبريكا بالكامل دواخل بحالة فو...,"EGP 380,000"
5,https://www.dubizzle.com.eg/en/ad/nissan-sunny...,Nissan,Sunny,95000,2022.0,Benzine,Automatic,NaN,Sedan,Descriptionللبيع نيسان صني 2022 حالة ممتازة جا...,"EGP 530,000"
6,https://www.dubizzle.com.eg/en/ad/%D8%B3%D8%AA...,Citroen,C3,550000,2004.0,Benzine,Manual,NaN,NaN,Descriptionعربيه لسه مغيره شورت بلوك ولسه راشش...,"EGP 230,000"
7,https://www.dubizzle.com.eg/en/ad/%D8%B3%D9%83...,Skoda,Kodiaq,22000,2023.0,Benzine,Automatic,1400.0,Estate,Descriptionوارد من الكويت \nتم الترخيص لمدة ٣ ...,"EGP 2,000,000"
8,https://www.dubizzle.com.eg/en/ad/%D8%B3%D9%88...,Suzuki,Swift,52000,2010.0,Benzine,Automatic,NaN,NaN,Descriptionالتواصل علي رقم \nاعلي فئه Burgman ...,"EGP 165,000"
9,https://www.dubizzle.com.eg/en/ad/%D8%AA%D9%88...,Toyota,Land Cruiser,42000,2023.0,Benzine,Automatic,3500.0,4X4,Descriptionلاندكروزر توين توربو خليجي اعلى موا...,"EGP 7,700,000"


In [5]:
df.info()   

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8519 entries, 0 to 8518
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   url                   8519 non-null   object 
 1   Brand                 8446 non-null   object 
 2   Model                 8390 non-null   object 
 3   Kilometers            8447 non-null   object 
 4   Year                  8448 non-null   float64
 5   Fuel Type             8448 non-null   object 
 6   Transmission Type     8448 non-null   object 
 7   Engine Capacity (CC)  6460 non-null   float64
 8   Body Type             7888 non-null   object 
 9   Description           8448 non-null   object 
 10  Price_EGP             8448 non-null   object 
dtypes: float64(2), object(9)
memory usage: 732.2+ KB


In [7]:
df.isnull().sum()

url                        0
Brand                     73
Model                    129
Kilometers                72
Year                      71
Fuel Type                 71
Transmission Type         71
Engine Capacity (CC)    2059
Body Type                631
Description               71
Price_EGP                 71
dtype: int64

In [ ]:
# we have our dataset ready we have some missing values we will handle them in the cleaning process
# also fix data types